# Figure 1 | Participant-level visual and semantic variance maps

Publication-style surface visualizations and cross-model consistency analyses for unique visual and unique semantic variance.

In [ ]:
import os
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import cortex

# Locate the analysis workspace. Set MITO_FMRI_ROOT explicitly when
# derivatives are stored outside this repository.
def find_analysis_root():
    configured = os.environ.get("MITO_FMRI_ROOT")
    candidates = [Path(configured)] if configured else []
    candidates += [Path.cwd(), Path.cwd() / "nsd_full_cortex", Path.cwd().parent / "nsd_full_cortex"]
    for candidate in candidates:
        if (candidate / "derivatives").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the analysis root. Set MITO_FMRI_ROOT to the nsd_full_cortex directory."
    )

ROOT = find_analysis_root()

SUBJECT = "subj07"
PYCORTEX_SUBJECT = "fsaverage"
N_VERTICES_HEMI = 163_842
MODELS = {
    "dinov2_minilm": "DINOv2–MiniLM",
    "cornet_s_mpnet": "CORnet-S–MPNet",
}
RESULTS = ROOT / "derivatives" / "encoding_significance" / "results" / SUBJECT
MASKS = ROOT / "derivatives" / "encoding_significance" / "masks"
RESPONSES = ROOT / "derivatives" / "encoding_significance" / "responses" / SUBJECT / "lh"
OUTPUT = ROOT / "derivatives" / "figure1" / "figures"
OUTPUT.mkdir(parents=True, exist_ok=True)

assert PYCORTEX_SUBJECT in cortex.db.subjects
left_points, _ = cortex.db.get_surf(PYCORTEX_SUBJECT, "fiducial", hemisphere="left")
assert len(left_points) == N_VERTICES_HEMI

In [ ]:
# Reconstruct full left-hemisphere fsaverage arrays and apply each model's significance mask.
vertex_indices = np.load(RESPONSES / "vertex_indices.npy")
surface_maps = {}
summary_rows = []

for model in MODELS:
    mask = np.load(MASKS / f"{SUBJECT}_{model}_encoding_mask.npz")["mask_lh"].astype(bool)
    for map_name in ("unique_visual", "unique_semantic"):
        values = np.load(RESULTS / model / "lh" / f"{map_name}.npy")
        full_lh = np.full(N_VERTICES_HEMI, np.nan, dtype=np.float32)
        full_lh[vertex_indices] = values
        full_lh[~mask] = np.nan
        surface_maps[(model, map_name)] = full_lh
        valid = full_lh[np.isfinite(full_lh)]
        summary_rows.append({
            "model": MODELS[model],
            "map": map_name,
            "n_vertices": len(valid),
            "median": float(np.median(valid)),
            "p99.5": float(np.quantile(valid, 0.995)),
            "max": float(valid.max()),
        })

# Scale the two dimensions separately, using a common range across models for each dimension.
VISUAL_MAX = float(np.quantile(np.concatenate([
    np.clip(surface_maps[(m, "unique_visual")][np.isfinite(surface_maps[(m, "unique_visual")])], 0, None)
    for m in MODELS
]), 0.995))
SEMANTIC_MAX = float(np.quantile(np.concatenate([
    np.clip(surface_maps[(m, "unique_semantic")][np.isfinite(surface_maps[(m, "unique_semantic")])], 0, None)
    for m in MODELS
]), 0.995))

print(f"Unique visual scale: 0–{VISUAL_MAX:.4f}")
print(f"Unique semantic scale: 0–{SEMANTIC_MAX:.4f}")
pd.DataFrame(summary_rows)

In [ ]:
# Complementary bivariate color mapping used in the manuscript figure.
VISUAL_RGB = np.array([35, 105, 200], dtype=float)      # blue
SEMANTIC_RGB = np.array([220, 50, 70], dtype=float)    # red
OVERLAP_RGB = np.array([65, 15, 105], dtype=float)    # purple
GAMMA = 1

VISUAL_MAX = 0.2
SEMANTIC_MAX = 0.04


# BEGIN BROAD ANATOMICAL LABELS
from matplotlib import patheffects as path_effects
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch

# Broad labels are descriptive landmarks, not an additional parcellation.
# Coordinates follow the pycortex fsaverage left-flatmap convention.
ANATOMICAL_LABELS = {
    "PFC": (-238.0, 42.0),
    "AC":  (-154.0, 4.0),
    "LTC": (-113.0, -66.0),
    "VTC": (-44.0, -103.0),
    "MPC": (-80.0, 83.0),
    "LPC": (-43.0, 31.0),
    "EVC": (-17.0, -20.0),
}


# Read the exact default sulcal paths from pycortex's fsaverage overlay.
# They are drawn directly with Matplotlib because Inkscape is unavailable.
pycortex_overlay = cortex.db.get_overlay(PYCORTEX_SUBJECT)
overlay_width, overlay_height = pycortex_overlay.svgshape
flat_merged, _ = cortex.db.get_surf(
    PYCORTEX_SUBJECT, "flat", merge=True, nudge=True
)
flat_min = np.asarray(flat_merged).min(axis=0)[:2]
flat_max = np.asarray(flat_merged).max(axis=0)[:2]


def add_pycortex_sulci(figure):
    """Draw pycortex's default fsaverage sulci without rasterizing via Inkscape."""
    ax = figure.axes[0]
    sulcus_effect = [
        path_effects.Stroke(linewidth=3.8, foreground="#777777", alpha=0.42),
        path_effects.Normal(),
    ]
    for shape in pycortex_overlay.layers["sulci"].shapes.values():
        for spline in shape.splines:
            vertices = spline.vertices.copy()
            vertices[:, 0] = (
                flat_min[0]
                + vertices[:, 0] / overlay_width * (flat_max[0] - flat_min[0])
            )
            vertices[:, 1] = (
                flat_max[1]
                - vertices[:, 1] / overlay_height * (flat_max[1] - flat_min[1])
            )
            patch = PathPatch(
                MplPath(vertices, spline.codes),
                facecolor="none",
                edgecolor="white",
                linewidth=5,
                alpha=0.68,
                capstyle="round",
                joinstyle="round",
                zorder=20,
            )
            patch.set_path_effects(sulcus_effect)
            ax.add_patch(patch)


def add_anatomical_labels(figure):
    """Add sparse, publication-style broad anatomical labels."""
    ax = figure.axes[0]
    text_effect = [
        path_effects.Stroke(linewidth=4.2, foreground="#4a4a4a", alpha=0.95),
        path_effects.Normal(),
    ]
    for label, position in ANATOMICAL_LABELS.items():
        ax.text(
            *position,
            label,
            color="#ffe500",
            fontsize=40,
            fontweight="bold",
            fontstyle="italic",
            ha="center",
            va="center",
            zorder=30,
            path_effects=text_effect,
        )
# END BROAD ANATOMICAL LABELS


def bivariate_rgba(visual, semantic):
    valid = np.isfinite(visual) & np.isfinite(semantic)
    v = np.zeros_like(visual, dtype=float)
    s = np.zeros_like(semantic, dtype=float)
    v[valid] = np.clip(visual[valid], 0, VISUAL_MAX) / VISUAL_MAX
    s[valid] = np.clip(semantic[valid], 0, SEMANTIC_MAX) / SEMANTIC_MAX
    v = v ** GAMMA
    s = s ** GAMMA

    # Four-corner bivariate interpolation. The explicit overlap corner makes
    # the low-low -> high-high diagonal perceptually ordered rather than nearly
    # constant gray.
    w_visual = v * (1 - s)
    w_semantic = (1 - v) * s
    w_overlap = v * s
    intensity = 1 - (1 - v) * (1 - s)
    rgb = np.zeros((visual.size, 3), dtype=float)
    active = intensity > 0
    rgb[active] = (
        w_visual[active, None] * VISUAL_RGB
        + w_semantic[active, None] * SEMANTIC_RGB
        + w_overlap[active, None] * OVERLAP_RGB
    ) / intensity[active, None]

    # Joint intensity increases monotonically from low-low to high-high while
    # retaining the cortical curvature beneath weak signals.
    alpha = np.zeros(visual.size, dtype=float)
    alpha[valid] = 255 * np.clip(intensity[valid], 0, 1)
    return (
        np.clip(rgb[:, 0], 0, 255).astype(np.uint8),
        np.clip(rgb[:, 1], 0, 255).astype(np.uint8),
        np.clip(rgb[:, 2], 0, 255).astype(np.uint8),
        np.clip(alpha, 0, 255).astype(np.uint8),
    )


def crop_left_flatmap(source: Path, destination: Path, padding: int = 24):
    image = Image.open(source).convert("RGBA")
    left = image.crop((0, 0, int(0.98*image.width // 2), image.height))
    rgb = np.asarray(left.convert("RGB"))
    content = np.any(rgb < 248, axis=2)
    yy, xx = np.where(content)
    box = (
        max(0, int(xx.min()) - padding), max(0, int(yy.min()) - padding),
        min(left.width, int(xx.max()) + padding + 1),
        min(left.height, int(yy.max()) + padding + 1),
    )
    left.crop(box).save(destination)


def render_bivariate(model: str) -> Path:
    visual = surface_maps[(model, "unique_visual")]
    semantic = surface_maps[(model, "unique_semantic")]
    r, g, b, a = bivariate_rgba(visual, semantic)
    # uint8 is essential here: pycortex otherwise rescales each RGB channel
    # independently and changes the intended orange/blue bivariate palette.
    zero_rh = np.zeros(N_VERTICES_HEMI, dtype=np.uint8)
    vertex = cortex.VertexRGB(
        np.concatenate([r, zero_rh]),
        np.concatenate([g, zero_rh]),
        np.concatenate([b, zero_rh]),
        PYCORTEX_SUBJECT,
        alpha=np.concatenate([a, zero_rh]),
    )
    figure = cortex.quickflat.make_figure(
        vertex,
        with_curvature=True,
        with_rois=False,
        with_labels=False,
        with_colorbar=False,
        with_borders=False,
        recache=False,
        nanmean=True,
        height=1600,
        curvature_brightness=0.7,
        curvature_contrast=0.15,
    )
    add_pycortex_sulci(figure)
    add_anatomical_labels(figure)
    with tempfile.TemporaryDirectory() as temporary:
        full_path = Path(temporary) / "bilateral.png"
        figure.savefig(full_path, dpi=200, bbox_inches="tight", pad_inches=0,
                       facecolor="white", transparent=False)
        plt.close(figure)
        output_path = OUTPUT / f"{SUBJECT}_{model}_unique_visual_semantic_lh_bivariate.png"
        crop_left_flatmap(full_path, output_path)
    return output_path

In [ ]:
# Export one bivariate cortical map for each encoding framework.
exported = {model: render_bivariate(model) for model in MODELS}
for path in exported.values():
    print(path.name)

# Export the bivariate key separately so that it cannot overlap the cortical panel.
n = 401
v = np.linspace(0, 1, n)
s = np.linspace(0, 1, n)
vv, ss = np.meshgrid(v, s)
vg, sg = vv ** GAMMA, ss ** GAMMA
w_visual = vg * (1 - sg)
w_semantic = (1 - vg) * sg
w_overlap = vg * sg
intensity = 1 - (1 - vg) * (1 - sg)
legend_rgb = np.ones((n, n, 3), dtype=float)
active = intensity > 0
mixed = np.zeros_like(legend_rgb)
mixed[active] = (
    w_visual[active, None] * VISUAL_RGB
    + w_semantic[active, None] * SEMANTIC_RGB
    + w_overlap[active, None] * OVERLAP_RGB
) / intensity[active, None]
alpha = intensity[..., None]
background = np.full_like(legend_rgb, 0.72)
legend_rgb = (alpha * mixed / 255 + (1 - alpha) * background).clip(0, 1)

fig, ax = plt.subplots(figsize=(3.1, 3.0))
ax.imshow(legend_rgb, origin="lower", extent=[0, VISUAL_MAX, 0, SEMANTIC_MAX], aspect="auto")
ax.set_box_aspect(1)
ax.set_xlabel(r"Visual", fontsize=26)
ax.set_ylabel(r"Semantic", fontsize=26)
ax.set_xticks([0, VISUAL_MAX]); ax.set_yticks([0, SEMANTIC_MAX])
ax.set_xticklabels(["0", f"{VISUAL_MAX:.2f}"])
ax.set_yticklabels(["0", f"{SEMANTIC_MAX:.2f}"])
ax.tick_params(labelsize=16)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
fig.tight_layout()
legend_path = OUTPUT / "subj01_unique_visual_semantic_bivariate_legend.png"
fig.savefig(legend_path, dpi=300, bbox_inches="tight", transparent=True)
plt.close(fig)
print(legend_path.name)

In [ ]:
# Display the final panels individually.
for model, path in exported.items():
    print(MODELS[model])
    display(Image.open(path))
print("Bivariate legend")
display(Image.open(legend_path))

## Left-hemisphere inflated-surface views

Lateral and posterior views use the same bivariate color mapping as the flatmap.

In [ ]:
# Render the fsaverage inflated surface off screen with VTK.
import vtk
from vtk.util.numpy_support import numpy_to_vtk, numpy_to_vtkIdTypeArray

SURFACE_3D = "inflated"
VIEW_SIZE = 1600
CURVATURE_BRIGHTNESS_3D = 0.70
CURVATURE_CONTRAST_3D = 0.15

inflated_lh, inflated_faces = cortex.db.get_surf(
    PYCORTEX_SUBJECT, SURFACE_3D, hemisphere="left"
)
curvature_lh = np.asarray(
    cortex.db.get_surfinfo(PYCORTEX_SUBJECT, type="curvature").data[:N_VERTICES_HEMI]
)


def vtk_polydata(points, faces):
    poly = vtk.vtkPolyData()
    vtk_points = vtk.vtkPoints()
    vtk_points.SetData(numpy_to_vtk(np.asarray(points, dtype=np.float32), deep=True))
    poly.SetPoints(vtk_points)

    packed = np.column_stack([
        np.full(len(faces), 3, dtype=np.int64),
        np.asarray(faces, dtype=np.int64),
    ]).ravel()
    cells = vtk.vtkCellArray()
    cells.SetCells(len(faces), numpy_to_vtkIdTypeArray(packed, deep=True))
    poly.SetPolys(cells)
    return poly


def composite_surface_rgb(model):
    visual = surface_maps[(model, "unique_visual")]
    semantic = surface_maps[(model, "unique_semantic")]
    r, g, b, a = bivariate_rgba(visual, semantic)
    overlay = np.column_stack([r, g, b]).astype(float)
    alpha = a.astype(float)[:, None] / 255.0

    # Match the thresholded gray curvature convention used by the flatmap.
    curvature_binary = (curvature_lh > 0).astype(float)
    gray = (
        (curvature_binary - 0.5) * CURVATURE_CONTRAST_3D
        + CURVATURE_BRIGHTNESS_3D
    )
    background = np.repeat((255 * gray[:, None]).clip(0, 255), 3, axis=1)
    return (alpha * overlay + (1 - alpha) * background).clip(0, 255).astype(np.uint8)


VIEW_SPECS = {
    "lateral":   (np.array([-1.0,  0.0,  0.0]), np.array([0.0,  0.0, 1.0])),
    "posterior": (np.array([ 0.0, -1.0,  0.0]), np.array([0.0,  0.0, 1.0])),
}


def crop_transparent_background(path, padding=24):
    image = Image.open(path).convert("RGBA")
    rgba = np.asarray(image)
    content = rgba[:, :, 3] > 0
    yy, xx = np.where(content)
    box = (
        max(0, int(xx.min()) - padding),
        max(0, int(yy.min()) - padding),
        min(image.width, int(xx.max()) + padding + 1),
        min(image.height, int(yy.max()) + padding + 1),
    )
    image.crop(box).save(path)


def render_3d_view(model, view_name):
    poly = vtk_polydata(inflated_lh, inflated_faces)
    colors = numpy_to_vtk(
        composite_surface_rgb(model),
        deep=True,
        array_type=vtk.VTK_UNSIGNED_CHAR,
    )
    colors.SetName("surface_rgb")
    colors.SetNumberOfComponents(3)
    poly.GetPointData().SetScalars(colors)

    normals = vtk.vtkPolyDataNormals()
    normals.SetInputData(poly)
    normals.SplittingOff()
    normals.ConsistencyOn()
    normals.AutoOrientNormalsOn()

    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputConnection(normals.GetOutputPort())
    mapper.SetColorModeToDirectScalars()
    mapper.SetScalarModeToUsePointData()
    mapper.InterpolateScalarsBeforeMappingOn()

    actor = vtk.vtkActor()
    actor.SetMapper(mapper)
    actor.GetProperty().SetAmbient(0.78)
    actor.GetProperty().SetDiffuse(0.22)
    actor.GetProperty().SetSpecular(0.0)

    renderer = vtk.vtkRenderer()
    renderer.SetBackground(1.0, 1.0, 1.0)
    renderer.SetBackgroundAlpha(0.0)
    renderer.AddActor(actor)

    window = vtk.vtkRenderWindow()
    window.SetOffScreenRendering(1)
    window.SetAlphaBitPlanes(1)
    window.SetSize(VIEW_SIZE, VIEW_SIZE)
    window.SetMultiSamples(8)
    window.AddRenderer(renderer)

    center = np.asarray(poly.GetCenter())
    bounds = np.asarray(poly.GetBounds()).reshape(3, 2)
    radius = float(np.max(bounds[:, 1] - bounds[:, 0]) / 2)
    direction, view_up = VIEW_SPECS[view_name]
    camera = renderer.GetActiveCamera()
    camera.SetFocalPoint(*center)
    camera.SetPosition(*(center + direction * radius * 4.0))
    camera.SetViewUp(*view_up)
    camera.ParallelProjectionOn()
    camera.SetParallelScale(radius * 1.08)
    renderer.ResetCameraClippingRange()

    window.Render()
    capture = vtk.vtkWindowToImageFilter()
    capture.SetInput(window)
    capture.SetInputBufferTypeToRGBA()
    capture.ReadFrontBufferOff()
    capture.Update()

    path = OUTPUT / f"{SUBJECT}_{model}_unique_visual_semantic_lh_{view_name}.png"
    writer = vtk.vtkPNGWriter()
    writer.SetFileName(str(path))
    writer.SetInputConnection(capture.GetOutputPort())
    writer.Write()
    window.Finalize()
    crop_transparent_background(path)
    return path

In [ ]:
# Export lateral and posterior transparent-background views for each model.
view_exports = {}
for model in MODELS:
    for view_name in VIEW_SPECS:
        path = render_3d_view(model, view_name)
        view_exports[(model, view_name)] = path
        with Image.open(path) as image:
            alpha = np.asarray(image.getchannel("A"))
            print(path.name, image.mode, f"transparent pixels={(alpha == 0).sum():,}")

In [ ]:
# Display lateral and posterior views for both encoding frameworks.
for model in MODELS:
    print(MODELS[model])
    for view_name in VIEW_SPECS:
        print(view_name.capitalize())
        display(Image.open(view_exports[(model, view_name)]))

## Univariate surface maps

Unique visual and unique semantic variance are displayed separately with fixed scales across views.

In [ ]:
UNIVARIATE_SPECS = {
    "unique_visual": {
        "rgb": VISUAL_RGB,
        "vmax": VISUAL_MAX,
        "label": r"Unique visual variance ($\Delta R^2$)",
    },
    "unique_semantic": {
        "rgb": SEMANTIC_RGB,
        "vmax": SEMANTIC_MAX,
        "label": r"Unique semantic variance ($\Delta R^2$)",
    },
}


def univariate_rgba(values, map_name):
    spec = UNIVARIATE_SPECS[map_name]
    valid = np.isfinite(values)
    strength = np.zeros(values.size, dtype=float)
    strength[valid] = (
        np.clip(values[valid], 0, spec["vmax"]) / spec["vmax"]
    ) ** GAMMA
    rgb = np.repeat(np.asarray(spec["rgb"], dtype=float)[None, :], values.size, axis=0)
    alpha = 255 * strength
    return (
        np.clip(rgb[:, 0], 0, 255).astype(np.uint8),
        np.clip(rgb[:, 1], 0, 255).astype(np.uint8),
        np.clip(rgb[:, 2], 0, 255).astype(np.uint8),
        np.clip(alpha, 0, 255).astype(np.uint8),
    )


def render_univariate_flatmap(model, map_name):
    values = surface_maps[(model, map_name)]
    r, g, b, a = univariate_rgba(values, map_name)
    zero_rh = np.zeros(N_VERTICES_HEMI, dtype=np.uint8)
    vertex = cortex.VertexRGB(
        np.concatenate([r, zero_rh]),
        np.concatenate([g, zero_rh]),
        np.concatenate([b, zero_rh]),
        PYCORTEX_SUBJECT,
        alpha=np.concatenate([a, zero_rh]),
    )
    figure = cortex.quickflat.make_figure(
        vertex,
        with_curvature=True,
        with_rois=False,
        with_labels=False,
        with_colorbar=False,
        with_borders=False,
        recache=False,
        nanmean=True,
        height=1600,
        curvature_brightness=0.7,
        curvature_contrast=0.15,
    )
    add_pycortex_sulci(figure)
    add_anatomical_labels(figure)
    with tempfile.TemporaryDirectory() as temporary:
        full_path = Path(temporary) / "bilateral.png"
        figure.savefig(
            full_path,
            dpi=200,
            bbox_inches="tight",
            pad_inches=0,
            facecolor="white",
            transparent=False,
        )
        plt.close(figure)
        output_path = OUTPUT / f"{SUBJECT}_{model}_{map_name}_lh_surface.png"
        crop_left_flatmap(full_path, output_path)
    return output_path


univariate_surface_exports = {
    (model, map_name): render_univariate_flatmap(model, map_name)
    for model in MODELS
    for map_name in UNIVARIATE_SPECS
}
for path in univariate_surface_exports.values():
    print(path.name)

In [ ]:
def display_preview(path, max_width=900):
    with Image.open(path) as image:
        preview = image.copy()
    if preview.width > max_width:
        preview.thumbnail((max_width, max_width), Image.Resampling.LANCZOS)
    display(preview)


for model in MODELS:
    print(MODELS[model])
    for map_name in UNIVARIATE_SPECS:
        print(UNIVARIATE_SPECS[map_name]["label"])
        display_preview(univariate_surface_exports[(model, map_name)])

## Univariate lateral and posterior views

In [ ]:
def composite_univariate_surface_rgb(model, map_name):
    values = surface_maps[(model, map_name)]
    r, g, b, a = univariate_rgba(values, map_name)
    overlay = np.column_stack([r, g, b]).astype(float)
    alpha = a.astype(float)[:, None] / 255.0

    curvature_binary = (curvature_lh > 0).astype(float)
    gray = (
        (curvature_binary - 0.5) * CURVATURE_CONTRAST_3D
        + CURVATURE_BRIGHTNESS_3D
    )
    background = np.repeat((255 * gray[:, None]).clip(0, 255), 3, axis=1)
    return (alpha * overlay + (1 - alpha) * background).clip(0, 255).astype(np.uint8)


def render_univariate_3d_view(model, map_name, view_name):
    poly = vtk_polydata(inflated_lh, inflated_faces)
    colors = numpy_to_vtk(
        composite_univariate_surface_rgb(model, map_name),
        deep=True,
        array_type=vtk.VTK_UNSIGNED_CHAR,
    )
    colors.SetName("surface_rgb")
    colors.SetNumberOfComponents(3)
    poly.GetPointData().SetScalars(colors)

    normals = vtk.vtkPolyDataNormals()
    normals.SetInputData(poly)
    normals.SplittingOff()
    normals.ConsistencyOn()
    normals.AutoOrientNormalsOn()

    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputConnection(normals.GetOutputPort())
    mapper.SetColorModeToDirectScalars()
    mapper.SetScalarModeToUsePointData()
    mapper.InterpolateScalarsBeforeMappingOn()

    actor = vtk.vtkActor()
    actor.SetMapper(mapper)
    actor.GetProperty().SetAmbient(0.78)
    actor.GetProperty().SetDiffuse(0.22)
    actor.GetProperty().SetSpecular(0.0)

    renderer = vtk.vtkRenderer()
    renderer.SetBackground(1.0, 1.0, 1.0)
    renderer.SetBackgroundAlpha(0.0)
    renderer.AddActor(actor)

    window = vtk.vtkRenderWindow()
    window.SetOffScreenRendering(1)
    window.SetAlphaBitPlanes(1)
    window.SetSize(VIEW_SIZE, VIEW_SIZE)
    window.SetMultiSamples(8)
    window.AddRenderer(renderer)

    center = np.asarray(poly.GetCenter())
    bounds = np.asarray(poly.GetBounds()).reshape(3, 2)
    radius = float(np.max(bounds[:, 1] - bounds[:, 0]) / 2)
    direction, view_up = VIEW_SPECS[view_name]
    camera = renderer.GetActiveCamera()
    camera.SetFocalPoint(*center)
    camera.SetPosition(*(center + direction * radius * 4.0))
    camera.SetViewUp(*view_up)
    camera.ParallelProjectionOn()
    camera.SetParallelScale(radius * 1.08)
    renderer.ResetCameraClippingRange()

    window.Render()
    capture = vtk.vtkWindowToImageFilter()
    capture.SetInput(window)
    capture.SetInputBufferTypeToRGBA()
    capture.ReadFrontBufferOff()
    capture.Update()

    path = OUTPUT / f"{SUBJECT}_{model}_{map_name}_lh_{view_name}.png"
    writer = vtk.vtkPNGWriter()
    writer.SetFileName(str(path))
    writer.SetInputConnection(capture.GetOutputPort())
    writer.Write()
    window.Finalize()
    crop_transparent_background(path)
    return path


univariate_view_exports = {
    (model, map_name, view_name): render_univariate_3d_view(
        model, map_name, view_name
    )
    for model in MODELS
    for map_name in UNIVARIATE_SPECS
    for view_name in VIEW_SPECS
}
for path in univariate_view_exports.values():
    with Image.open(path) as image:
        alpha = np.asarray(image.getchannel("A"))
        print(path.name, image.mode, f"transparent pixels={(alpha == 0).sum():,}")

In [ ]:
for model in MODELS:
    print(MODELS[model])
    for map_name in UNIVARIATE_SPECS:
        print(UNIVARIATE_SPECS[map_name]["label"])
        for view_name in VIEW_SPECS:
            print(view_name.capitalize())
            display_preview(univariate_view_exports[(model, map_name, view_name)], 700)

## Univariate color keys

In [ ]:
def render_univariate_colorbar(map_name):
    spec = UNIVARIATE_SPECS[map_name]
    n = 512
    strength = np.linspace(0, 1, n) ** GAMMA
    target = np.asarray(spec["rgb"], dtype=float) / 255.0
    background = np.full(3, 0.72)
    gradient = (
        strength[:, None] * target[None, :]
        + (1 - strength[:, None]) * background[None, :]
    )

    fig, ax = plt.subplots(figsize=(4, 1))
    ax.imshow(
        gradient[None, :, :],
        aspect="auto",
        extent=[0, spec["vmax"], 0, 1],
        origin="lower",
    )
    ax.set_yticks([])
    ax.set_xticks([0, spec["vmax"]])
    ax.set_xticklabels(["0", f"{spec['vmax']:.2f}"], size=30)
    #ax.set_xlabel(spec["label"])
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    fig.tight_layout(pad=0.35)
    path = OUTPUT / f"{SUBJECT}_{map_name}_colorbar.png"
    fig.savefig(path, dpi=300, bbox_inches="tight", transparent=True)
    plt.close(fig)
    return path


univariate_colorbars = {
    map_name: render_univariate_colorbar(map_name)
    for map_name in UNIVARIATE_SPECS
}
for map_name, path in univariate_colorbars.items():
    print(path.name)
    display(Image.open(path))

## Cross-model consistency of unique variance

In [ ]:
from scipy.stats import pearsonr
from matplotlib.lines import Line2D

CORR_SUBJECTS = ("subj01", "subj02", "subj05", "subj07")
CORR_MAPS = {
    "unique_visual": {
        "label": "Unique visual variance",
        "color": "#2369C8",
        "marker": "o",
    },
    "unique_semantic": {
        "label": "Unique semantic variance",
        "color": "#DC3246",
        "marker": "s",
    },
}
ALL_RESULTS = ROOT / "derivatives" / "encoding_significance" / "results"
ALL_RESPONSES = ROOT / "derivatives" / "encoding_significance" / "responses"
CORR_OUTPUT = OUTPUT


def cross_model_values(subject, model, map_name):
    mask_data = np.load(MASKS / f"{subject}_{model}_encoding_mask.npz")
    value_parts, mask_parts = [], []
    for hemi in ("lh", "rh"):
        vertex_idx = np.load(ALL_RESPONSES / subject / hemi / "vertex_indices.npy")
        values = np.load(ALL_RESULTS / subject / model / hemi / f"{map_name}.npy")
        significant = mask_data[f"mask_{hemi}"][vertex_idx].astype(bool)
        value_parts.append(np.asarray(values, dtype=np.float64))
        mask_parts.append(significant)
    return np.concatenate(value_parts), np.concatenate(mask_parts)


cross_model_data = {}
correlation_rows = []
for subject in CORR_SUBJECTS:
    for map_name in CORR_MAPS:
        x, x_mask = cross_model_values(subject, "dinov2_minilm", map_name)
        y, y_mask = cross_model_values(subject, "cornet_s_mpnet", map_name)
        keep = x_mask & y_mask & np.isfinite(x) & np.isfinite(y)
        x_keep, y_keep = x[keep], y[keep]
        r = float(pearsonr(x_keep, y_keep).statistic)
        cross_model_data[(subject, map_name)] = (x_keep, y_keep)
        correlation_rows.append({
            "subject": subject,
            "map": map_name,
            "pearson_r": r,
            "n_intersection_vertices": int(keep.sum()),
            "hemispheres": "lh+rh",
        })

cross_model_correlations = pd.DataFrame(correlation_rows)
cross_model_correlations.to_csv(
    CORR_OUTPUT / "cross_model_unique_variance_correlations.csv", index=False
)
cross_model_correlations

## Vertex-wise cross-model correlations in the displayed participant

In [ ]:
from matplotlib.ticker import FormatStrFormatter

rng = np.random.default_rng(20260805)
subject_corr_exports = {}
CORR_AXIS_SPECS = {
    "unique_visual": {
        "limits": (-0.05, 0.50),
        "ticks": [0, 0.10, 0.20, 0.30, 0.40, 0.50],
    },
    "unique_semantic": {
        "limits": (-0.012, 0.12),
        "ticks": [0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12],
    },
}

for map_name, spec in CORR_MAPS.items():
    fig, ax = plt.subplots(figsize=(5.0, 5.0), constrained_layout=True)
    x, y = cross_model_data[(SUBJECT, map_name)]
    r = cross_model_correlations.loc[
        cross_model_correlations["subject"].eq(SUBJECT)
        & cross_model_correlations["map"].eq(map_name), "pearson_r"
    ].iloc[0]
    n = len(x)
    shown = np.arange(n) if n <= 10_000 else rng.choice(n, 10_000, replace=False)
    ax.scatter(
        x[shown], y[shown], s=7, alpha=0.20, color=spec["color"],
        edgecolors="none", rasterized=True,
    )

    lo, hi = CORR_AXIS_SPECS[map_name]["limits"]
    ticks = CORR_AXIS_SPECS[map_name]["ticks"]
    slope, intercept = np.polyfit(x, y, 1)
    xx = np.array([lo, hi])
    ax.plot(xx, slope * xx + intercept, color="#35125A", lw=2.0)
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xticks=ticks, yticks=ticks)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(spec["label"], fontsize=16, pad=10)
    ax.set_xlabel("DINOv2 + MiniLM", fontsize=14)
    ax.set_ylabel("CORnet-S + MPNet", fontsize=14)
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.text(
        0.72, 0.16, f"$r$ = {r:.3f}\n$p$ < $.0001$",
        transform=ax.transAxes, ha="left", va="top", fontsize=12,
        bbox={"boxstyle": "round,pad=0.28", "facecolor": "white",
              "edgecolor": "0.78", "linewidth": 0.8, "alpha": 0.92},
    )
    ax.tick_params(labelsize=12)
    ax.spines[["top", "right"]].set_visible(False)

    stem = f"{SUBJECT}_cross_model_{map_name}_correlation"
    png_path = CORR_OUTPUT / f"{stem}.png"
    pdf_path = CORR_OUTPUT / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    subject_corr_exports[map_name] = {"png": png_path, "pdf": pdf_path}

for map_name in CORR_MAPS:
    display(Image.open(subject_corr_exports[map_name]["png"]))

## Cross-model correlations across all four participants

In [ ]:
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(5, 4.0), constrained_layout=True)
ax.set_aspect("auto")
x_centers = {"unique_visual": 0.0, "unique_semantic": 1.0}
subject_offsets = dict(zip(CORR_SUBJECTS, (-0.045, -0.015, 0.015, 0.045)))
subject_markers = dict(zip(CORR_SUBJECTS, ("o", "s", "^", "D")))

for subject in CORR_SUBJECTS:
    subject_rows = (
        cross_model_correlations[cross_model_correlations["subject"].eq(subject)]
        .set_index("map")
    )
    r_visual = float(subject_rows.loc["unique_visual", "pearson_r"])
    r_semantic = float(subject_rows.loc["unique_semantic", "pearson_r"])
    offset = subject_offsets[subject]
    """ax.plot(
        [x_centers["unique_visual"] + offset,
         x_centers["unique_semantic"] + offset],
        [r_visual, r_semantic],
        color="0.78", linewidth=1.2, zorder=1,
    )"""

    for map_name, value in (
        ("unique_visual", r_visual),
        ("unique_semantic", r_semantic),
    ):
        spec = CORR_MAPS[map_name]
        ax.scatter(
            x_centers[map_name] + offset, value,
            s=90, color=spec["color"], marker=subject_markers[subject],
            edgecolor="white", linewidth=0.9, zorder=3,
        )

ax.set_xticks(
    [x_centers["unique_visual"], x_centers["unique_semantic"]],
    ["unique visual\nvariance", "unique semantic\nvariance"],
)
ax.set_xlim(-0.35, 2.35)
ax.set_ylim(0, 1.01)
ax.set_ylabel("Cross-model Pearson $r$", fontsize=14)
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
#ax.grid(axis="y", color="0.90", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
subject_legend_handles = [
    Line2D(
        [0], [0], linestyle="none", marker=subject_markers[subject],
        markerfacecolor="black", markeredgecolor="black", markersize=8,
        label=subject.replace("subj", "NSD Sub-"),
    )
    for subject in CORR_SUBJECTS
]
ax.legend(
    handles=subject_legend_handles,
    ncol=1, loc="lower right",
    #bbox_to_anchor=(0.5, 1.01),
    #handletextpad=0.45, columnspacing=1.2,
)

group_corr_png = CORR_OUTPUT / "four_subject_cross_model_unique_variance_correlations_paired_dots.png"
group_corr_pdf = group_corr_png.with_suffix(".pdf")
fig.savefig(group_corr_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(group_corr_pdf, bbox_inches="tight", facecolor="white")
plt.close(fig)
display(Image.open(group_corr_png))

display(
    cross_model_correlations.pivot(index="subject", columns="map", values="pearson_r")
    .rename(columns={key: value["label"] for key, value in CORR_MAPS.items()})
    .round(3)
)